# Try first test run for pixelgen - CITE-seq cross integration

## This notebook is outdated (paths broken) but should work if paths are updated (authored by Florian)

In [ ]:
import anndata as ad
import scanpy as sc
import pandas as pd
import cytovi
import matplotlib.pyplot as plt

In [ ]:
pixel_dir = "/home/labs/amit/floriani/Lab/PROJECTS/Pixelgen/data/raw/PBMC_pilot_data/"
adata_pxl = ad.read_h5ad(f'{pixel_dir}/V2_combined_data_filtered_normalized.h5ad')
adata_pxl.obs['technology'] = 'pixelgen'

adata_cite = ad.read_h5ad(f'{pixel_dir}/CITE_seq_seuratv4_dsb_normalized.h5ad')
adata_cite.obs['technology'] = 'cite_seq'

In [ ]:
# merge data and scale
adata = cytovi.pp.merge_batches([adata_pxl, adata_cite], scaled_layer_key='dsb')
cytovi.pp.scale(adata, transformed_layer_key='dsb', batch_key = 'technology')

# compute embedding from data space

In [ ]:
# subset on bb markers
bb_markers = adata.var_names[(adata.var['_batch_0'] & adata.var['_batch_1'])]
adata_bb = adata[:, bb_markers].copy()

adata_bb.X = adata_bb.layers['scaled']
sc.tl.pca(adata_bb)
sc.pp.neighbors(adata_bb)
sc.tl.umap(adata_bb)
adata_bb.obsm['X_umap_unintegrated'] = adata_bb.obsm['X_umap']
sc.pl.umap(adata_bb, color='technology')

# Train cytoVI model

In [ ]:
# train model
cytovi.CytoVI.setup_anndata(adata, layer='scaled', batch_key='batch')
model = cytovi.CytoVI(adata, protein_likelihood='normal')
model.train(batch_size=1024)

In [ ]:
# check convergance
plt.subplot(1, 2, 1)
plt.plot(model.history['elbo_train'])
plt.xlabel('epochs')
plt.ylabel('elbo_train')

plt.subplot(1, 2, 2)
plt.plot(model.history['elbo_validation'])
plt.xlabel('epochs')
plt.ylabel('elbo_validation')

In [ ]:
# get latent space
adata.obsm['X_CytoVI'] = model.get_latent_representation()

# impute missing markers
adata.layers['imputed_cross'] = model.get_normalized_expression(n_samples = 10)

sc.pp.neighbors(adata, use_rep='X_CytoVI')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['technology'])

In [ ]:
cytovi.pl.histogram(adata, marker = [*bb_markers], layer_key='scaled', groupby='technology')

# without mog

In [ ]:
# train model
cytovi.CytoVI.setup_anndata(adata, layer='scaled', batch_key='batch')
model = cytovi.CytoVI(adata, protein_likelihood='normal', prior_mixture = False)
model.train(batch_size=1024)

In [ ]:
# check convergance
plt.subplot(1, 2, 1)
plt.plot(model.history['elbo_train'])
plt.xlabel('epochs')
plt.ylabel('elbo_train')

plt.subplot(1, 2, 2)
plt.plot(model.history['elbo_validation'])
plt.xlabel('epochs')
plt.ylabel('elbo_validation')

In [ ]:
# get latent space
adata.obsm['X_CytoVI'] = model.get_latent_representation()

# impute missing markers
adata.layers['imputed_cross'] = model.get_normalized_expression(n_samples = 10)

sc.pp.neighbors(adata, use_rep='X_CytoVI')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['technology'])

# repeat but use arcsinh transformation

In [ ]:
cytovi.pp.arcsinh(adata_pxl)
cytovi.pp.scale(adata_pxl)

cytovi.pp.arcsinh(adata_cite)
cytovi.pp.scale(adata_cite)

adata = cytovi.pp.merge_batches([adata_pxl, adata_cite])

In [ ]:
# subset on bb markers
bb_markers = adata.var_names[(adata.var['_batch_0'] & adata.var['_batch_1'])]
adata_bb = adata[:, bb_markers].copy()

adata_bb.X = adata_bb.layers['scaled']
sc.tl.pca(adata_bb)
sc.pp.neighbors(adata_bb)
sc.tl.umap(adata_bb)
adata_bb.obsm['X_umap_unintegrated'] = adata_bb.obsm['X_umap']
sc.pl.umap(adata_bb, color='technology')

In [ ]:
# train model
cytovi.CytoVI.setup_anndata(adata, layer='scaled', batch_key='batch')
model = cytovi.CytoVI(adata, protein_likelihood='normal')
model.train(batch_size=1024)

In [ ]:
# check convergance
plt.subplot(1, 2, 1)
plt.plot(model.history['elbo_train'])
plt.xlabel('epochs')
plt.ylabel('elbo_train')

plt.subplot(1, 2, 2)
plt.plot(model.history['elbo_validation'])
plt.xlabel('epochs')
plt.ylabel('elbo_validation')

In [ ]:
# get latent space
adata.obsm['X_CytoVI'] = model.get_latent_representation()

# impute missing markers
adata.layers['imputed_cross'] = model.get_normalized_expression(n_samples = 10)

sc.pp.neighbors(adata, use_rep='X_CytoVI')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['technology'])

# repeat without mog

In [ ]:
# train model
cytovi.CytoVI.setup_anndata(adata, layer='scaled', batch_key='batch')
model = cytovi.CytoVI(adata, protein_likelihood='normal', prior_mixture=False)
model.train(batch_size=1024)

In [ ]:
# check convergance
plt.subplot(1, 2, 1)
plt.plot(model.history['elbo_train'])
plt.xlabel('epochs')
plt.ylabel('elbo_train')

plt.subplot(1, 2, 2)
plt.plot(model.history['elbo_validation'])
plt.xlabel('epochs')
plt.ylabel('elbo_validation')

In [ ]:
adata_pxl

In [ ]:
adata_cite

In [ ]:
# get latent space
adata.obsm['X_CytoVI'] = model.get_latent_representation()

# impute missing markers
adata.layers['imputed_cross'] = model.get_normalized_expression(n_samples = 10)

sc.pp.neighbors(adata, use_rep='X_CytoVI')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['technology'])

# isotropic gaussian + adv scale

In [ ]:
# train model
cytovi.CytoVI.setup_anndata(adata, layer='scaled', batch_key='batch')
model = cytovi.CytoVI(adata, protein_likelihood='normal', prior_mixture=False)
model.train(batch_size=1024, plan_kwargs={'scale_adversarial_loss': 2})

In [ ]:
# check convergance
plt.subplot(1, 2, 1)
plt.plot(model.history['elbo_train'])
plt.xlabel('epochs')
plt.ylabel('elbo_train')

plt.subplot(1, 2, 2)
plt.plot(model.history['elbo_validation'])
plt.xlabel('epochs')
plt.ylabel('elbo_validation')

In [ ]:
# get latent space
adata.obsm['X_CytoVI'] = model.get_latent_representation()

# impute missing markers
adata.layers['imputed_cross'] = model.get_normalized_expression(n_samples = 10)

sc.pp.neighbors(adata, use_rep='X_CytoVI')
sc.tl.umap(adata)
sc.pl.umap(adata, color=['technology'])

In [ ]:
adata.obs

In [ ]:
[*adata.var_names[~adata.var['_batch_0'] & adata.var['_batch_1']]]

In [ ]:
adata.var.loc[['CD3', 'CD4', 'CD8', 'CD19', 'CD14', 'CD56'], :]

In [ ]:
sc.pl.umap(adata, color=['CD4', 'CD8', 'CD19', 'CD14'], layer='imputed_cross', cmap='mako', ncols=2)

In [ ]:
sc.pl.umap(adata, color=['CD56', 'IgM', 'CD1c', 'XCR1'], layer='imputed_cross', cmap='mako', ncols=2)